<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/ClinicalAssistaint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install --upgrade torchao peft

In [9]:
# Install libraries for Fine-Tuning, PEFT (LoRA/QLoRA), HuggingFace Transformers, and Gemini SDK
!pip install -q -U google-genai transformers datasets peft trl accelerate bitsandbytes

import os
import torch
from google.colab import userdata
from google.genai import types, client

# Retrieve Gemini API Key from Google Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

# Initialize Gemini Client
ai = client.Client(api_key=api_key)

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

print("Setup complete! Fine-tuning dependencies and Gemini client initialized.")

CUDA Available: True
Device Name: Tesla T4
Setup complete! Fine-tuning dependencies and Gemini client initialized.


In [10]:
from datasets import Dataset

# 1. Create raw clinical notes paired with concise target medical summaries
raw_clinical_data = [
    {
        "instruction": "Summarize the following raw clinical note into a structured medical summary containing Chief Complaint, Key Findings, Diagnosis, and Treatment Plan.",
        "input": """Patient John Doe, 54M, came in today complaining of acute onset chest tightness and shortness of breath that started 3 hours ago during mild exertion. Patient has a history of dyslipidemia and hypertension. BP is elevated at 155/95 mmHg, pulse rate 92 bpm, spO2 96% on room air. ECG showed ST-segment depression in V4-V6. Troponin I elevated at 0.8 ng/mL. Administered Aspirin 325mg and sublingual Nitroglycerin in clinic with moderate symptom relief. Referred to Cardiology for urgent coronary angiography.""",
        "output": """Chief Complaint: Acute chest tightness and shortness of breath.
Key Findings: History of dyslipidemia & hypertension. BP 155/95 mmHg, HR 92 bpm. ECG shows ST depression (V4-V6), Troponin I elevated at 0.8 ng/mL.
Diagnosis: Non-ST-Elevation Myocardial Infarction (NSTEMI).
Treatment Plan: Administered Aspirin 325mg and sublingual Nitroglycerin. Urgent referral to Cardiology for coronary angiography."""
    },
    {
        "instruction": "Summarize the following raw clinical note into a structured medical summary containing Chief Complaint, Key Findings, Diagnosis, and Treatment Plan.",
        "input": """Jane Smith, 32F, reports severe right lower quadrant abdominal pain accompanied by nausea and low-grade fever (100.8 F) for the last 18 hours. Physical exam reveals positive McBurney's point tenderness and rebound tenderness. Lab results show leukocytosis with WBC 14,500/mcL. Abdominal ultrasound confirms an enlarged, non-compressible appendix measuring 8.5mm with surrounding fluid. Patient kept NPO, started on IV normal saline and IV Ceftriaxone. Surgical consultation requested for emergency appendectomy.""",
        "output": """Chief Complaint: Right lower quadrant abdominal pain, nausea, and fever.
Key Findings: Fever 100.8 F, positive McBurney's and rebound tenderness. WBC 14,500/mcL. Ultrasound confirms inflamed appendix (8.5mm).
Diagnosis: Acute Appendicitis.
Treatment Plan: NPO status, IV fluid hydration, IV Ceftriaxone. Emergency surgical consultation for appendectomy."""
    },
    {
        "instruction": "Summarize the following raw clinical note into a structured medical summary containing Chief Complaint, Key Findings, Diagnosis, and Treatment Plan.",
        "input": """Robert Taylor, 68M, presents with worsening productive cough with rust-colored sputum, high fever (102.4 F), and right-sided pleuritic chest pain over the past 3 days. On examination, bronchial breath sounds and crackles heard over the right lower lung lobe. Chest X-ray reveals right lower lobe consolidation consistent with bacterial pneumonia. Oxygen saturation 91% on room air. Started on supplemental O2 via nasal cannula at 2L/min, IV Levofloxacin 750mg daily, and admitted to the medical ward.""",
        "output": """Chief Complaint: Productive cough with rust-colored sputum, high fever, and pleuritic chest pain.
Key Findings: Fever 102.4 F, SpO2 91% RA. Right lower lung crackles. CXR shows right lower lobe consolidation.
Diagnosis: Community-Acquired Bacterial Pneumonia.
Treatment Plan: Supplemental O2 (2L nasal cannula), IV Levofloxacin 750mg daily. Admitted to medical ward."""
    }
]

# 2. Convert to Hugging Face Dataset format
dataset = Dataset.from_list(raw_clinical_data)

# 3. Format into a unified Alpaca/Instruction prompt structure
def format_instruction_prompt(sample):
    return f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""

# Map formatted text to the dataset
formatted_texts = [format_instruction_prompt(sample) for sample in dataset]
dataset = dataset.add_column("formatted_text", formatted_texts)

print("Dataset prepared successfully!")
print(f"Total clinical samples: {len(dataset)}\n")
print("=== SAMPLE FORMATTED PROMPT ===")
print(dataset[0]["formatted_text"])

Dataset prepared successfully!
Total clinical samples: 3

=== SAMPLE FORMATTED PROMPT ===
### Instruction:
Summarize the following raw clinical note into a structured medical summary containing Chief Complaint, Key Findings, Diagnosis, and Treatment Plan.

### Input:
Patient John Doe, 54M, came in today complaining of acute onset chest tightness and shortness of breath that started 3 hours ago during mild exertion. Patient has a history of dyslipidemia and hypertension. BP is elevated at 155/95 mmHg, pulse rate 92 bpm, spO2 96% on room air. ECG showed ST-segment depression in V4-V6. Troponin I elevated at 0.8 ng/mL. Administered Aspirin 325mg and sublingual Nitroglycerin in clinic with moderate symptom relief. Referred to Cardiology for urgent coronary angiography.

### Response:
Chief Complaint: Acute chest tightness and shortness of breath.
Key Findings: History of dyslipidemia & hypertension. BP 155/95 mmHg, HR 92 bpm. ECG shows ST depression (V4-V6), Troponin I elevated at 0.8 ng/m

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

# 1. Target Base Model ID
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading Base Model and Tokenizer: {MODEL_ID}...")

# 2. Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Load Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

# 4. Configure LoRA (Parameter-Efficient Fine-Tuning)
peft_config = LoraConfig(
    r=8,                       # Rank matrix dimension
    lora_alpha=16,             # Scaling factor
    lora_dropout=0.05,         # Dropout probability
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"]  # Target attention layers
)

# 5. Define SFTConfig
sft_config = SFTConfig(
    output_dir="./clinical_assistant_lora",
    dataset_text_field="formatted_text",
    max_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

# 6. Initialize Supervised Fine-Tuning (SFT) Trainer
# Note: Pass base_model directly; SFTTrainer attaches peft_config automatically
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    args=sft_config,
    peft_config=peft_config,
)

print("\nStarting simulated fine-tuning step...")
trainer.train()
print("\nModel fine-tuning simulation complete!")

Loading Base Model and Tokenizer: Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting simulated fine-tuning step...


Step,Training Loss
1,1.763623
2,1.943235



Model fine-tuning simulation complete!


In [17]:
# 1. Prepare an unseen clinical note for testing
unseen_clinical_note = """Patient Sarah Jenkins, 45F, presents with severe right-sided throbbing headache for 24 hours, accompanied by photophobia, phonophobia, and nausea. Patient reports similar past episodes diagnosed as migraines without aura. Vitals: BP 122/78 mmHg, HR 76 bpm, Temp 98.6 F. Neurological examination is non-focal. Administered Sumatriptan 6mg subcutaneous and Metoclopramide 10mg IV with complete pain resolution within 90 minutes. Prescribed oral Rizatriptan 10mg PRN for future acute attacks and advised headache diary tracking."""

# 2. Format as instruction prompt
eval_prompt = f"""### Instruction:
Summarize the following raw clinical note into a structured medical summary containing Chief Complaint, Key Findings, Diagnosis, and Treatment Plan.

### Input:
{unseen_clinical_note}

### Response:
"""

# 3. Tokenize input
inputs = tokenizer(eval_prompt, return_tensors="pt").to(model.device)

# 4. Generate summary with fine-tuned model
print("Generating medical summary for unseen clinical note...\n")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )

# 5. Decode and display generated summary
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
response_only = generated_text.split("### Response:\n")[-1]

print("=== GENERATED CLINICAL SUMMARY ===")
print(response_only.strip())

Generating medical summary for unseen clinical note...

=== GENERATED CLINICAL SUMMARY ===
Chief Complaint: Severe right-sided throbbing headache for 24 hours, accompanied by photophobia, phonophobia, and nausea.
Key Findings: 
- Vitals: Blood pressure (BP) 122/78 mmHg, heart rate (HR) 76 bpm, body temperature (Temp) 98.6°F
- Neurological examination: Non-focal
- Administered medication: Sumatriptan 6mg subcutaneous and Metoclopramide 10mg IV
- Pain resolution: Complete within 90 minutes
- Prescription: Oral Rizatriptan 10mg PRN for future acute attacks

Diagnosis: Migraine with aura
Treatment Plan: 

1. Sumatriptan 6mg subcutaneous
2. Metoclopramide 10mg IV
3. Rizatriptan 10mg PRN
4. Headache diary


In [27]:
# 1. Define prompt asking Gemini to explain healthcare fine-tuning trade-offs
interpretation_prompt = """
You are an expert AI Engineer and Medical Technology Consultant.

Explain and compare the following 4 fine-tuning methods for adaptation of Large Language Models in healthcare clinical note summarization:
1. Full Fine-Tuning
2. LoRA (Low-Rank Adaptation)
3. QLoRA (Quantized Low-Rank Adaptation)
4. DoRA (Weight-Decomposed Low-Rank Adaptation)

Provide two clear structural explanations:

---
### PART 1: BEGINNER-FRIENDLY EXPLANATION (Analogies & Simple Terms)
Explain how each method works using real-world analogies (editing textbooks, sticky notes, memory usage, training speed) for hospital administrators.

---
### PART 2: EXPERT TECHNICAL COMPARISON (For AI Engineers)
Provide a structured comparison matrix covering:
- Trainable Parameter Percentage (%)
- GPU VRAM / Compute Memory Requirements
- Training Latency & Efficiency
- Risk of Catastrophic Forgetting
- Suitability for HIPAA-Compliant On-Premise Deployments
"""

# 2. Automatically select an available Gemini Flash model
available_models = [m.name for m in ai.models.list()]
flash_model = next((m for m in available_models if "flash" in m), "gemini-2.5-flash")

# Remove leading 'models/' prefix if present for generate_content
model_id = flash_model.replace("models/", "")
print(f"Using active Gemini model: {model_id}\n")

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=1500
)

# 3. Call Gemini
response = ai.models.generate_content(
    model=model_id,
    contents=interpretation_prompt,
    config=config
)

print("=== GEMINI INTERPRETATION & TRADE-OFF ANALYSIS ===")
print(response.text)

Using active Gemini model: gemini-2.5-flash



ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}